# **Data Cleaning Notebook**

## Purpose

This notebook transforms the merged raw country-year dataset into a clean, analysis-ready dataset.

The goal is to resolve data quality issues that remain after the raw setup stage, including missing values, redundant variables, inconsistent data types, country coverage limitations, and variables that require careful treatment before exploratory data analysis.

This notebook also introduces supplementary country classification and geographic metadata to enrich the analytical dataset.

The output of this notebook is the cleaned dataset saved in `data/processed/`.

## Project Context

The project investigates sustainable wellbeing across countries by combining subjective wellbeing, inequality, economic development, emissions, energy use, material footprint, and country classification variables.

Because the dataset is built from multiple public sources with different structures, coverage, and definitions, a transparent cleaning process is necessary before substantive EDA can be conducted.

## Input Datasets

This notebook starts from the merged raw dataset created in the raw setup notebook:

```text
data/processed_raw/global_sustainability_wellbeing_resource_data_raw.csv
```

This file already combines the primary source datasets into a common country-year structure, but it may still contain missing values, redundant variables, inconsistent types, and countries with insufficient analytical coverage.

This notebook also uses supplementary files from:

```text
data/supplementary/
```

These supplementary data are introduced at the cleaning stage to add contextual classification and geographic information, especially where some development-region classifications are not applicable to all countries.

## Main Tasks

This notebook performs the following cleaning steps:

1. Load the merged raw dataset.
2. Inspect dataset shape, columns, data types, and missing values.
3. Resolve redundant or conflicting variables created during merging.
4. Assess missingness by variable, country, and year.
5. Define criteria for retaining or excluding countries.
6. Apply justified imputations where appropriate.
7. Recode non-substantive missing values where missingness means “not applicable.”
8. Add supplementary country classification and geographic metadata.
9. Create or correct ranking variables where necessary.
10. Convert variables to appropriate data types.
11. Validate the final cleaned dataset.
12. Export the cleaned dataset to `data/processed/`.

## Key Cleaning Decisions to Document

The cleaning process should explicitly document decisions such as:

- Why the final analysis window is restricted to 2013–2021.
- Which countries are removed due to insufficient data coverage.
- How missing happiness values are treated.
- How missing GINI values are interpolated.
- Why some UNDP development-region values are recoded as not applicable.
- How redundant merge columns are resolved.
- Which variables are converted from float to integer and why.
- Which supplementary variables are added and what analytical role they serve.

## Checks to Add or Maintain

The notebook should include explicit validation checks for:

- Duplicate `country`–`year` observations before and after cleaning.
- Required variables after each major cleaning stage.
- Remaining missing values after imputation or recoding.
- Data type consistency before export.
- Expected country, year, row, and column counts.
- Successful reload of the exported cleaned dataset.

## Output

The output of this notebook is:

```text
data/processed/sustainability_wellbeing_resource_data_clean.csv
```

This file is the final analysis-ready dataset used in the EDA notebook.

We load the merged raw *Global Wellbeing, Sustainability and Resource Use Dataset*. 

We screen the data and look for missing values, specifically trying to identify countries with a substantial amount of missing values in the key variables:

| Key variables | Description | Source(s) |
|---|---|---|
| `happiness_index` | Measure of Subjective Wellbeing | World Happiness Report (https://www.kaggle.com/datasets/simonaasm/world-happiness-index-by-reports-2013-2023) |
| `gini_index` | Measure of income inequality | World Bank GINI Index (https://data.worldbank.org/indicator/SI.POV.GINI) |
| `consumption_co2_per_capita` | Consumption-based CO₂ emissions per capita | Our World in Data CO₂ (https://github.com/owid/co2-data) |
| `co2_per_capita` | Production-based CO₂ emissions per capita | Our World in Data CO₂ Data (https://github.com/owid/co2-data) |
| `renewables_consumption` | Share of primary energy consumption from renewable sources | Our World in Data Energy (https://github.com/owid/energy-data) |
| `energy_per_capita` | Primary energy consumption per capita | Our World in Data Energy (https://github.com/owid/energy-data) (See below) |
| `material_footprint_per_capita` | Per capita material consiumption indicator | UN Human Development Reports (https://www.kaggle.com/datasets/iamsouravbanerjee/material-footprint-per-capita-by-country) |

This will help us identify countries that may be more reasonable to drop than keep and handle missing values of. Relative to our variables of interest they contain more noise than actual information.

## 1. Duplicate columns from different origin datasts

In [1]:
import pandas as pd

In [2]:
# Make imports from the scr/ directory work.
import sys
from pathlib import Path

# Add project root to path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

In [3]:
from src.io import load_csv
from src.config import MERGED_RAW_PATH

df_raw = load_csv(MERGED_RAW_PATH)
display(df_raw.head())

df_raw.info()

,country,iso_code,year,co2_per_capita,consumption_co2_per_capita,energy_per_capita_x,temperature_change_from_co2,share_global_co2,land_use_change_co2_per_capita,population,...,renewables_consumption,happiness_index,happiness_index_rank,continent,hemisphere,human_development_groups,hdi_rank_2021,undp_developing_regions,material_footprint_per_capita,gini_index
0,Aruba,ABW,2013,8.395,NaN,47742.637,0.0,0.002,NaN,102570.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Aruba,ABW,2014,8.435,NaN,47990.926,0.0,0.002,NaN,103381.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Aruba,ABW,2015,8.615,NaN,48905.531,0.0,0.003,NaN,104200.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Aruba,ABW,2016,8.411,NaN,47619.418,0.0,0.002,NaN,104989.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Aruba,ABW,2017,8.420,NaN,49061.195,0.0,0.002,NaN,105737.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 1970 entries, 0 to 1969
Data columns (total 22 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   country                         1970 non-null   str    
 1   iso_code                        1970 non-null   str    
 2   year                            1970 non-null   int64  
 3   co2_per_capita                  1925 non-null   float64
 4   consumption_co2_per_capita      1079 non-null   float64
 5   energy_per_capita_x             1844 non-null   float64
 6   temperature_change_from_co2     1943 non-null   float64
 7   share_global_co2                1943 non-null   float64
 8   land_use_change_co2_per_capita  1754 non-null   float64
 9   population                      1853 non-null   float64
 10  gdp                             1484 non-null   float64
 11  energy_per_capita_y             1844 non-null   float64
 12  renewables_consumption          711 non-null 

We notice we have two energy_per_capita variables, _x and _y. These come from two different datasets preceding the merge into a single dataset.
1. We consider both to establish whether they are different in any way.
2. If different, we must establish a criterion for keeping one over the other - first consider if any of the two has more missing values.

In [4]:
print(df_raw['energy_per_capita_x'].isna().sum())
print(df_raw['energy_per_capita_y'].isna().sum())

# We now identify which countries and years have missing data in each variable

print(df_raw[df_raw['energy_per_capita_x'].isna()]['country'].unique())
print(df_raw[df_raw['energy_per_capita_y'].isna()]['country'].unique())

# Final check that the non-missing values are the same in both variables, so we can safely drop one of them.
# We do this by computing their differences in every row, storing the results into a set, and then checking if the set contains only zero.

print(set(df_raw[df_raw['energy_per_capita_x'].notna()]['energy_per_capita_x'] - df_raw[df_raw['energy_per_capita_y'].notna()]['energy_per_capita_y']))

# There seem to be some differences. 
# We must find which values are different between the two variables 
# and print them as a list indexed by country and year 
# in order to understand where the differences are.

# Find rows where both values exist AND they're different
mask = (df_raw['energy_per_capita_x'].notna() & 
        df_raw['energy_per_capita_y'].notna() & 
        (df_raw['energy_per_capita_x'] != df_raw['energy_per_capita_y']))

print(df_raw[mask][['iso_code', 'year', 'energy_per_capita_x', 'energy_per_capita_y']])


126
126
<StringArray>
[                       'Anguilla',                         'Andorra',
                      'Antarctica', 'Bonaire Sint Eustatius and Saba',
                         'Curacao',                'Christmas Island',
                   'Liechtenstein',                          'Monaco',
                'Marshall Islands',                           'Palau',
                      'San Marino',       'Sint Maarten (Dutch part)',
                         'Vatican',               'Wallis and Futuna']
Length: 14, dtype: str
<StringArray>
[                       'Anguilla',                         'Andorra',
                      'Antarctica', 'Bonaire Sint Eustatius and Saba',
                         'Curacao',                'Christmas Island',
                   'Liechtenstein',                          'Monaco',
                'Marshall Islands',                           'Palau',
                      'San Marino',       'Sint Maarten (Dutch part)',
                  

So all differences come from a single country, Togo (iso_code: TGO), where `energy_per_capita_y > energy_per_capita_x`

Looking at the source of the original datasets, the energy dataset (y) has been updated more recently than the CO2 dataset (x) - 3 weeks vs 5 months. This aligns with the observation prior to merging, that for the same countries, the energy dataset had greater richness in the population and gdp variables - which where consequently kept to maximise the availability of real information.

Following along these lines, we make the choice to keep the y-variable, from energy data, rather than taking an average between the two.

In [5]:
# We must now rename the energy_per_capita_y variable to energy_per_capita, and drop the _x version
df_raw = df_raw.rename(columns={'energy_per_capita_y': 'energy_per_capita'})
df_raw = df_raw.drop(columns=['energy_per_capita_x'])

## 2. Identify and deal with low Signal-to-Noise ratio countries in the dataset - Reduction of unnecessary dimensionality (rows)

Looking at missing values per year shows how many countries are missing per variable per year.

In [6]:
key_vars = [
    "happiness_index", 
    "gini_index", 
    "material_footprint_per_capita", 
    "consumption_co2_per_capita", 
    "co2_per_capita", 
    "energy_per_capita", 
    "renewables_consumption"
    ]

missing_by_year = (
    df_raw[key_vars]
    .isna()
    .groupby(df_raw['year'])
    .sum()
    )

display(missing_by_year)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
year,,,,,,,
2013,67,143,61,99,5,14,140
2014,218,135,61,98,5,14,139
2015,65,133,61,99,5,14,140
2016,66,137,61,100,5,14,140
2017,66,142,61,99,5,14,140
2018,65,127,61,99,5,14,140
2019,66,142,61,99,5,14,140
2020,69,150,61,99,5,14,140
2021,73,139,61,99,5,14,140


We select key variables and identify the number of missing values per iso_code/country (where there are any) aggregated over the years.

To aid with this process in a managable manner given the large number of countries, we build a function to:
1. Compute missing values by country-year.
2. Identify country-years with more than N missing variables.
3. Extract the affected countries.
4. Return the country-level missingness summary for only those countries.

In [7]:
from src.cleaning import countries_with_missing_vars

key_vars = [
    "happiness_index", 
    "gini_index", 
    "material_footprint_per_capita", 
    "consumption_co2_per_capita", 
    "co2_per_capita", 
    "energy_per_capita", 
    "renewables_consumption"
    ]

subset_missing_6 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=6
)

display(subset_missing_6)

# We can see that the only countries with missing data are Antarctica, Christmas Island, Congo, Monaco, San Marino and Vatican.
# We can drop these countries from our dataset by first finding their iso_code from df_raw and then dropping them from the dataframe

iso_codes_to_drop = df_raw[df_raw['country'].isin(subset_missing_6.index)]['iso_code'].unique()
df_raw = df_raw[~df_raw['iso_code'].isin(iso_codes_to_drop)]

# Confirm that the countries with missing data have been dropped
new_subset_missing_6 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=6
)

display(new_subset_missing_6)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,
Antarctica,9,9,9,9,9,9,9
Christmas Island,9,9,9,9,9,9,9
Congo,9,17,0,17,0,0,17
Monaco,9,9,9,9,9,9,9
San Marino,9,9,9,9,9,9,9
Vatican,9,9,9,9,9,9,9


,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,


We now look for countries with missing values for 6 of the 7 key variables.

We see that 9 countries, out of all the key variables, only have observations for all years for co2_per_capita. This is not enough to justify keeping them in the dataset.

In [8]:
subset_missing_5 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=5
)

display(subset_missing_5)

iso_codes_to_drop = df_raw[df_raw['country'].isin(subset_missing_5.index)]['iso_code'].unique()
df_raw = df_raw[~df_raw['iso_code'].isin(iso_codes_to_drop)]

new_subset_missing_5 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=5
)

display(new_subset_missing_5)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,
Andorra,9,9,9,9,0,9,9
Anguilla,9,9,9,9,0,9,9
Bonaire Sint Eustatius and Saba,9,9,9,9,0,9,9
Curacao,9,9,9,9,0,9,9
Liechtenstein,9,9,9,9,0,9,9
Marshall Islands,9,8,9,9,0,9,9
Palau,9,9,9,9,0,9,9
Sint Maarten (Dutch part),9,9,9,9,0,9,9
Wallis and Futuna,9,9,9,9,0,9,9


,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,


We now look for countries with missing values for 5 of the 7 key variables using the function we built again.

- 42 of the remaining countries only have values for co2_per_capita and energy_per_capita, for all years consistently in all cases.
- Some of the countries have some sparse readings for happiness and gini indices, which means that at best they have valid entries for 3 of the 7 key variables, and only for some years. This is less than 50%, so we risk introducing more noise than information into the data.
- The decision is made to drop all the identified countries on the basis of too many missing values with a consistent pattern that is not recoverable in a meaningful manner.


In [9]:
subset_missing_4 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=4
)

display(subset_missing_4)

iso_codes_to_drop = df_raw[df_raw['country'].isin(subset_missing_4.index)]['iso_code'].unique()
df_raw = df_raw[~df_raw['iso_code'].isin(iso_codes_to_drop)]

new_subset_missing_4 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=4
)

display(new_subset_missing_4)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,
Antigua and Barbuda,9,9,9,9,0,0,9
Aruba,9,9,9,9,0,0,9
Barbados,9,8,9,9,0,0,9
Bermuda,9,9,9,9,0,0,9
British Virgin Islands,9,9,9,9,0,0,9
Cape Verde,9,8,9,9,0,0,9
Comoros,3,7,9,9,0,0,9
Cook Islands,9,9,9,9,0,0,9
Dominica,9,9,9,9,0,0,9


,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,


In [10]:
print(f"There are {df_raw['country'].nunique()} countries in the dataset.")

There are 161 countries in the dataset.


We check for countries missing 4/7 key vars.

Here the search becomes more challenging, the main pattern that can be observed is:
- All identified countries consistently have all values for energy_per_capita and co2_per_capita, and in most cases material_footrpint_per_capita.
- Many of them only have one missing year in the happiness_index variable. But this was the case for all countries:
    - To aid with navigating this more nuanced scenario, we first impute the missing values in year 2014 in the happiness index variable.

In [11]:
subset_missing_3 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=3
)

display(subset_missing_3)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,
Afghanistan,1,9,0,9,0,0,9
Angola,4,8,0,9,0,0,9
Bahamas,9,9,0,9,0,0,9
Belize,6,8,0,9,0,0,9
Bhutan,4,8,0,9,0,0,9
Bosnia and Herzegovina,1,7,0,9,0,0,9
Burundi,1,7,0,9,0,0,9
Central African Republic,3,8,0,9,0,0,9
Chad,1,8,0,9,0,0,9


In [12]:
key_vars = [
    "happiness_index", 
    "gini_index", 
    "material_footprint_per_capita", 
    "consumption_co2_per_capita", 
    "co2_per_capita", 
    "energy_per_capita", 
    "renewables_consumption"
    ]

missing_by_year = (
    df_raw[key_vars]
    .isna()
    .groupby(df_raw['year'])
    .sum()
    )

display(missing_by_year)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
year,,,,,,,
2013,14,89,5,41,0,0,82
2014,161,80,5,41,0,0,82
2015,12,78,5,41,0,0,82
2016,12,83,5,42,0,0,82
2017,11,86,5,41,0,0,82
2018,10,72,5,41,0,0,82
2019,12,89,5,41,0,0,82
2020,16,93,5,41,0,0,82
2021,20,83,5,41,0,0,82


### 2.1 Interpolation: Happiness index missing 2014 for all countries

We note that all 2014 values for happiness index are missing. We interpolate these by country linearly between the previous and next years to 2014.

In [13]:
# Check how many 2014 missing values were filled
print(f"2014 missing before: {df_raw[(df_raw['year'] == 2014) & (df_raw['happiness_index'].isnull())].shape[0]}")

# Create a Series with interpolated values for all rows
interpolated = df_raw.groupby('iso_code')['happiness_index'].transform(lambda x: x.interpolate(method='linear'))

# Only fill 2014 missing values
mask_2014 = (df_raw['year'] == 2014) & (df_raw['happiness_index'].isnull())
df_raw.loc[mask_2014, 'happiness_index'] = interpolated[mask_2014]


print(f"2014 missing after: {df_raw[(df_raw['year'] == 2014) & (df_raw['happiness_index'].isnull())].shape[0]}")

2014 missing before: 161
2014 missing after: 14


In [14]:
subset_missing_3 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=3
)

display(subset_missing_3)


,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,
Angola,3,8,0,9,0,0,9
Bahamas,9,9,0,9,0,0,9
Belize,6,8,0,9,0,0,9
Bhutan,4,8,0,9,0,0,9
Central African Republic,2,8,0,9,0,0,9
Cuba,9,9,0,9,0,0,9
Democratic Republic of Congo,1,8,0,9,0,0,9
Djibouti,6,7,0,9,0,0,9
Equatorial Guinea,9,9,0,9,0,0,9


This interpolation nearly halved the number of countries missing 4/7 key vars.

Since these countries are missing consumption_co2_per_capita and most entries for happiness_index and gini_index, we do not have enough data to look at the desired features for these countries.

In [15]:
iso_codes_to_drop = df_raw[df_raw['country'].isin(subset_missing_3.index)]['iso_code'].unique()
df_raw = df_raw[~df_raw['iso_code'].isin(iso_codes_to_drop)]

new_subset_missing_3 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=3
)
display(new_subset_missing_3)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,


In [16]:
print(f"There are {df_raw['country'].nunique()} countries in the dataset.")

There are 144 countries in the dataset.


In [17]:
key_vars = [
    "happiness_index", 
    "gini_index", 
    "material_footprint_per_capita", 
    "consumption_co2_per_capita", 
    "co2_per_capita", 
    "energy_per_capita", 
    "renewables_consumption"
    ]

subset_missing_2 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=2
)

display(subset_missing_2)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,
Afghanistan,0,9,0,9,0,0,9
Bosnia and Herzegovina,0,7,0,9,0,0,9
Brunei,9,9,0,0,0,0,9
Burundi,0,7,0,9,0,0,9
Chad,0,8,0,9,0,0,9
Cote d'Ivoire,2,6,0,0,0,0,9
Gabon,0,8,0,9,0,0,9
Haiti,0,9,0,9,0,0,9
Laos,1,8,0,0,0,0,9


Material consumption and consumption co2 are central to the analysis of interest.

We can safaly drop countries that are still at this point missing vlues for all years of both these columns.

We do the same criterion but look for countries that are missing more than 3 years in either happiness index or gini_index - since these are the key social indicators.

In [18]:
# Step 1: Find rows in subset_missing_2 where both variables are missing
key_missing = subset_missing_2[
    ((subset_missing_2['material_footprint_per_capita']==9) & 
    (subset_missing_2['consumption_co2_per_capita']==9))
    |
    ((subset_missing_2['happiness_index']>3) |
    (subset_missing_2['gini_index']>3))
]

# Step 2: Get unique iso_code from these rows
countries_to_drop = df_raw[df_raw['country'].isin(key_missing.index)]['iso_code'].unique()

# Step 3: Display countries to be dropped
print(f"Countries to drop: {countries_to_drop}")
print(f"Number of countries: {len(countries_to_drop)}")

# Step 4: Remove these countries from df_raw
df_raw = df_raw[~df_raw['iso_code'].isin(countries_to_drop)]

# Step 5: Verify removal
print(f"Remaining unique countries: {df_raw['iso_code'].nunique()}")

Countries to drop: <StringArray>
['AFG', 'BDI', 'BIH', 'BRN', 'CIV', 'GAB', 'HTI', 'LAO', 'LBN', 'LBR', 'LBY',
 'MLI', 'MMR', 'MNE', 'MOZ', 'MRT', 'MUS', 'NAM', 'NER', 'SLE', 'TCD', 'TTO',
 'YEM']
Length: 23, dtype: str
Number of countries: 23
Remaining unique countries: 121


In [19]:
subset_missing_2 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=2
)
display(subset_missing_2)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,


All countries missing all gini_index entries are also missing either all consumption co2 or material footprint data. We can safely drop these as they are missing too many key components.

In [20]:
# Step 1: Find rows in subset_missing_2 where all gini_index entries are missing
gini_missing = subset_missing_2[
    (subset_missing_2['gini_index']==9)
    & ((subset_missing_2['consumption_co2_per_capita']==9)
       | (subset_missing_2['material_footprint_per_capita']==9))
]

# Step 2: Get unique iso_code from these rows
countries_to_drop = df_raw[df_raw['country'].isin(gini_missing.index)]['iso_code'].unique()

# Step 3: Display countries to be dropped
print(f"Countries to drop: {countries_to_drop}")
print(f"Number of countries: {len(countries_to_drop)}")

# Step 4: Remove these countries from df_raw
df_raw = df_raw[~df_raw['iso_code'].isin(countries_to_drop)]

# Step 5: Verify removal
print(f"Remaining unique countries: {df_raw['iso_code'].nunique()}")

Countries to drop: <StringArray>
[]
Length: 0, dtype: str
Number of countries: 0
Remaining unique countries: 121


In [21]:
key_vars = [
    "happiness_index", 
    "gini_index", 
    "material_footprint_per_capita",
    "consumption_co2_per_capita", 
    "renewables_consumption"
    ]

subset_missing_2 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=2
)
display(subset_missing_2)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,renewables_consumption
country,,,,,


### 2.2. Final crop: Countries missing all years for key variables.

At this point we determine that any county missing all entries for the key variables should be dropped. Missing values that offer some point of reference in at least one year can be extrapolated in some way, but no reference point means that this country does not have enough info to be of real value to this dataset. Since all the above countries contain missing values for all years in the renewables_consumption variable, we drop these too.

In [22]:
key_vars = [
    "happiness_index", 
    "gini_index", 
    "material_footprint_per_capita", 
    "consumption_co2_per_capita", 
    "co2_per_capita", 
    "energy_per_capita", 
    "renewables_consumption"
]

# Find iso_codes where ANY key variable is missing for all 9 years
iso_to_drop = (
    df_raw
    .groupby("iso_code")[key_vars]
    .apply(lambda x: (x.isna().sum() == 9).any())
)

# Keep only iso_codes that do NOT satisfy the condition
df_raw = df_raw[
    ~df_raw["iso_code"].isin(iso_to_drop[iso_to_drop].index)
]

print(f"Remaining unique countries: {df_raw['iso_code'].nunique()}")

Remaining unique countries: 62


This leaves only Qatar as the final country with missing values for more than 1 key variable. So the process of removing countries without any observations for some key variable also helped in this way.

QAT is kept since, while it misses 2 entries in hapiness_index and only has gini_index for one year, it has all observations for material, impact and socioeconomic data of interest for all but 2 of the key variables represented. Happiness index missing values can be carefully imputed to reliably match historical and adjacent country trends.

gini_index for QAT will be imputed but this will be carefully taken into account to avoind unrealistic conclusions due to these imputed missing values.

In [23]:
subset_missing_1 = countries_with_missing_vars(
    df=df_raw,
    key_vars=key_vars,
    thresh_missing_vars=1
)
display(subset_missing_1)

,happiness_index,gini_index,material_footprint_per_capita,consumption_co2_per_capita,co2_per_capita,energy_per_capita,renewables_consumption
country,,,,,,,
Qatar,2,8,0,0,0,0,0


In [24]:
print(f"Remaining unique countries: {df_raw['iso_code'].nunique()}")

# The remaining countries are:
display(df_raw['country'].unique())
df_raw.info()

Remaining unique countries: 62


<StringArray>
['United Arab Emirates',            'Argentina',            'Australia',
              'Austria',              'Belgium',           'Bangladesh',
             'Bulgaria',              'Belarus',               'Brazil',
               'Canada',          'Switzerland',                'Chile',
                'China',             'Colombia',               'Cyprus',
              'Czechia',              'Germany',              'Denmark',
              'Ecuador',                'Egypt',                'Spain',
              'Estonia',              'Finland',               'France',
       'United Kingdom',               'Greece',              'Croatia',
              'Hungary',            'Indonesia',              'Ireland',
                 'Iran',               'Israel',                'Italy',
                'Japan',           'Kazakhstan',          'South Korea',
            'Sri Lanka',            'Lithuania',           'Luxembourg',
               'Latvia',             

<class 'pandas.DataFrame'>
Index: 558 entries, 54 to 1951
Data columns (total 21 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   country                         558 non-null    str    
 1   iso_code                        558 non-null    str    
 2   year                            558 non-null    int64  
 3   co2_per_capita                  558 non-null    float64
 4   consumption_co2_per_capita      558 non-null    float64
 5   temperature_change_from_co2     558 non-null    float64
 6   share_global_co2                558 non-null    float64
 7   land_use_change_co2_per_capita  558 non-null    float64
 8   population                      558 non-null    float64
 9   gdp                             558 non-null    float64
 10  energy_per_capita               558 non-null    float64
 11  renewables_consumption          558 non-null    float64
 12  happiness_index                 556 non-null    fl

### 2.3 Some remarks on the resulting country coverage

We have substantial geopolitical and developmental heterogeneity:

- OECD core: Japan, Germany, Nordics, Netherlands, etc.
- BRICS-adjacent/emerging blocs: China, Brazil, Russia, South Africa.
- Southeast Asia: Indonesia, Thailand, Vietnam, Malaysia, Philippines.
- MENA representation: Egypt, Morocco, Qatar, UAE, Iran.
- Latin America coverage is decent.
- Eastern Europe/post-Soviet coverage is also solid.

The main structural limitation is underrepresentation of:

- Sub-Saharan Africa (outside South Africa),
- low-income economies,
- small developing states,
- and highly fragile economies.

In [25]:
# At this point we export data_raw as a .csv file with name sustainability_wellbeing_resource_data_missing_values.csv so anyone can deal with missing data any way they consider fit.
df_raw.to_csv('../data/processed/sustainability_wellbeing_resource_data_missing_values.csv')

## 3. Missing values - Happiness index

In [26]:
df_raw.loc[
    (df_raw["happiness_index"].isna()),
    ["country", "iso_code", "year"]
].sort_values(["country", "year"])

,country,iso_code,year
1464,Qatar,QAT,2020
1465,Qatar,QAT,2021


We want to deal with these missing values in a manner that considers:
- In country historic tendencies.
- Potential substantial shocks that may have induced substantial differences with historical tendencies (e.g. COVID, since missing values are 2020, 2021)
    - Geographically (continent) and developmentally (human dev) similar countries can help account for this.

Let's print the differences in happiness index for QAT for available years sequentially backwards into a list to see how different the values are relative to the range of the available differences.

Then let's calculate the average only for countries with the same continent and human_development_groups as `iso_code==QAT` of the same change in happiness_index (from one year to the next, from 2013 to 2019). We use this to screen for similarity and significant shocks in recent years.



In [27]:
# QAT sequential backward differences in happiness_index
qat_diff = (
    df_raw.loc[df_raw["iso_code"] == "QAT", ["year", "happiness_index"]]
    .sort_values("year", ascending=False)
    .assign(
        diff=lambda x: x["happiness_index"].diff(-1)
    )
)

# Get QAT continent and HDI group
qat_meta = (
    df_raw.loc[df_raw["iso_code"] == "QAT",
               ["continent", "human_development_groups"]]
    .drop_duplicates()
    .iloc[0]
)

qat_continent = qat_meta["continent"]
qat_hdi = qat_meta["human_development_groups"]

# Countries matching QAT continent + HDI group
peer_countries = (
    df_raw.loc[
        (df_raw["continent"] == qat_continent) &
        (df_raw["human_development_groups"] == qat_hdi),
        "iso_code"
    ]
    .unique()
)

# Average yearly sequential backward differences for peer countries
peer_avg_diff = (
    df_raw.loc[df_raw["iso_code"].isin(peer_countries),
               ["iso_code", "year", "happiness_index"]]
    .sort_values(["iso_code", "year"], ascending=[True, False])
    .groupby("iso_code")
    .apply(
        lambda x: x.assign(
            diff=x["happiness_index"].diff(-1)
        )
    )
    .reset_index(drop=True)
    .groupby("year")["diff"]
    .mean()
    .reset_index(name="peer_avg_diff")
)

# Merge side-by-side
comparison = (
    qat_diff[["year", "diff"]]
    .rename(columns={"diff": "QAT_diff"})
    .merge(peer_avg_diff, on="year", how="left")
)

# Create readable year ranges
comparison["year_range"] = (
    comparison["year"].astype(str)
    + "-"
    + (comparison["year"] - 1).astype(str)
)

# Add difference column
comparison["difference"] = (
    comparison["QAT_diff"] - comparison["peer_avg_diff"]
)

# Reorder columns
comparison = comparison[
    ["year_range", "QAT_diff", "peer_avg_diff", "difference"]
]

print(comparison.to_string(index=False))



year_range  QAT_diff  peer_avg_diff  difference
 2021-2020       NaN      -0.022222         NaN
 2020-2019       NaN       0.008333         NaN
 2019-2018    0.0000      -0.086300     0.08630
 2018-2017   -0.0010       0.011500    -0.01250
 2017-2016    0.0000       0.013800    -0.01380
 2016-2015   -0.2360      -0.055800    -0.18020
 2015-2014   -0.0275      -0.059650     0.03215
 2014-2013   -0.0275      -0.059650     0.03215
 2013-2012       NaN            NaN         NaN


We look at the ratio of variance for the differences in happiness_index between QAT and peer countries.

In [28]:
# QAT sequential backward differences
qat_diff = (
    df_raw.loc[df_raw["iso_code"] == "QAT", ["year", "happiness_index"]]
    .sort_values("year", ascending=False)
    .assign(
        diff=lambda x: x["happiness_index"].diff(-1)
    )["diff"]
    .dropna()
)

# Get QAT continent and HDI group
qat_meta = (
    df_raw.loc[df_raw["iso_code"] == "QAT",
               ["continent", "human_development_groups"]]
    .drop_duplicates()
    .iloc[0]
)

qat_continent = qat_meta["continent"]
qat_hdi = qat_meta["human_development_groups"]

# Peer countries
peer_countries = (
    df_raw.loc[
        (df_raw["continent"] == qat_continent) &
        (df_raw["human_development_groups"] == qat_hdi),
        "iso_code"
    ]
    .unique()
)

# Sequential backward differences for peer countries
peer_diffs = (
    df_raw.loc[df_raw["iso_code"].isin(peer_countries),
               ["iso_code", "year", "happiness_index"]]
    .sort_values(["iso_code", "year"], ascending=[True, False])
    .groupby("iso_code")
    .apply(
        lambda x: x.assign(
            diff=x["happiness_index"].diff(-1)
        )
    )
    .reset_index(drop=True)["diff"]
    .dropna()
)

# Variances
qat_variance = qat_diff.var()
peer_variance = peer_diffs.var()

# Comparison
variance_comparison = pd.DataFrame({
    "group": ["QAT", "Peers"],
    "variance": [qat_variance, peer_variance]
})

variance_comparison["variance_ratio_vs_peers"] = (
    variance_comparison["variance"] / peer_variance
)

print(variance_comparison.to_string(index=False))

group  variance  variance_ratio_vs_peers
  QAT  0.008600                 0.305359
Peers  0.028163                 1.000000


We impute missing happiness_index values for QAT (for years 2020 and 2021) by adding to the previous year's value a scaled peer deviation. Specifically, we:

1. Calculate the peer countries' average year-to-year change (difference) for each target year
2. Convert this peer change into a z-score relative to the distribution of all peer yearly differences (mean-centered and normalized by their standard deviation)
3. Scale this z-score by the variance ratio (QAT variance / peer variance) to adjust for differences in volatility between QAT and peer countries
4. Transform the scaled z-score back to QAT's native scale using QAT's own historical mean and standard deviation of yearly differences
5. Add this calibrated change to QAT's previous year's actual value to produce the imputed value

It converts the peer difference to a standardized score (z-score), scales that score by the variance ratio, then converts back to QAT's scale using QAT's own difference distribution parameters. This preserves the relative position (percentile rank) of the peer change while adapting it to QAT's typical magnitude and variability of changes.



In [29]:
# 1. QAT available year-to-year diffs
qat_diffs = (
    df_raw.loc[df_raw["iso_code"] == "QAT",
               ["year", "happiness_index"]]
    .sort_values("year")
    .assign(
        diff=lambda x: x["happiness_index"].diff()
    )
)

# Keep observed diffs only
qat_observed_diffs = qat_diffs["diff"].dropna()

# QAT diff statistics
qat_diff_mean = qat_observed_diffs.mean()
qat_diff_std = qat_observed_diffs.std()

# 2. Peer yearly average diffs
peer_yearly = (
    df_raw.loc[df_raw["iso_code"].isin(peer_countries),
               ["iso_code", "year", "happiness_index"]]
    .sort_values(["iso_code", "year"])
    .groupby("iso_code")
    .apply(
        lambda x: x.assign(
            diff=x["happiness_index"].diff()
        )
    )
    .reset_index(drop=True)
)

# Mean and std of peer diffs by year
peer_diff_stats = (
    peer_yearly
    .groupby("year")["diff"]
    .agg(["mean", "std"])
    .rename(columns={
        "mean": "peer_mean_diff",
        "std": "peer_std_diff"
    })
)

# 3. Variance ratio
variance_ratio = qat_variance / peer_variance

# 4. Sequential imputation
qat_series = (
    df_raw.loc[df_raw["iso_code"] == "QAT",
               ["year", "happiness_index"]]
    .sort_values("year")
    .copy()
)

for yr in [2020, 2021]:

    prev_year = yr - 1

    prev_value = qat_series.loc[
        qat_series["year"] == prev_year,
        "happiness_index"
    ].iloc[0]

    # Peer average change for this year
    peer_change = peer_diff_stats.loc[yr, "peer_mean_diff"]

    # Peer std for this year
    peer_std = peer_diff_stats.loc[yr, "peer_std_diff"]


    # Convert peer change to variance units
    peer_z = (
        (peer_change - peer_diff_stats["peer_mean_diff"].mean())
        / peer_diff_stats["peer_mean_diff"].std()
    )

    # Scale variance units by variance ratio
    qat_scaled_z = peer_z * variance_ratio

    # Transform back to QAT-specific scale
    qat_change = (
        qat_diff_mean +
        qat_scaled_z * qat_diff_std
    )

    # Imputed value
    imputed_value = prev_value + qat_change

    # Insert
    qat_series.loc[
        qat_series["year"] == yr,
        "happiness_index"
    ] = imputed_value

# -----------------------------
# 5. View final series
# -----------------------------
print(qat_series.to_string(index=False))

 year  happiness_index
 2013         6.666000
 2014         6.638500
 2015         6.611000
 2016         6.375000
 2017         6.375000
 2018         6.374000
 2019         6.374000
 2020         6.353933
 2021         6.311788


In [30]:
# Merge imputed values back to df_raw
for yr in [2020, 2021]:
    imputed_val = qat_series.loc[qat_series["year"] == yr, "happiness_index"].iloc[0]
    df_raw.loc[(df_raw["iso_code"] == "QAT") & (df_raw["year"] == yr), "happiness_index"] = imputed_val

print(f"There are {df_raw['happiness_index'].isna().sum()} missing values in happiness_index column")

There are 0 missing values in happiness_index column


## 4. Missing values - Happiness Index Rank

We check for remining missing values

In [31]:
df_prepared = df_raw.copy()
df_prepared.info()

for x in df_prepared.columns:
    if df_prepared[x].isna().sum() > 0: 
        print("\nVariable:", x, "\nMissing values:", df_prepared[x].isna().sum())

<class 'pandas.DataFrame'>
Index: 558 entries, 54 to 1951
Data columns (total 21 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   country                         558 non-null    str    
 1   iso_code                        558 non-null    str    
 2   year                            558 non-null    int64  
 3   co2_per_capita                  558 non-null    float64
 4   consumption_co2_per_capita      558 non-null    float64
 5   temperature_change_from_co2     558 non-null    float64
 6   share_global_co2                558 non-null    float64
 7   land_use_change_co2_per_capita  558 non-null    float64
 8   population                      558 non-null    float64
 9   gdp                             558 non-null    float64
 10  energy_per_capita               558 non-null    float64
 11  renewables_consumption          558 non-null    float64
 12  happiness_index                 558 non-null    fl

In [32]:
missing_by_year = (
    df_raw['happiness_index_rank']
    .isna()
    .groupby(df_raw['year'])
    .sum()
    )

display(missing_by_year)

year
2013     0
2014    62
2015     0
2016     0
2017     0
2018     0
2019     0
2020     1
2021     1
Name: happiness_index_rank, dtype: int64

This variable came from the same dataset as happiness_index. Therefore we are missing entries for all countries on 2014 here too. Additionally, we have the QAT missing values in 2020 and 2021, as expected.

One option is going back, filling missing happiness_index values like before and running into further issues with imputation due to missing values at 2013 - not allowing for interpolation, and the imputed ranks reproducing the structure of missing values. Instead, we build a rank specific to this dataset happiness_index_rank_62, which is an integer that ranks each country within each year by happiness_index (which already contains all observations).

This was also a float, so we ensure the new rank is an integer.

The function add_sample_yearly_rank uses rank method=min, hence allowing ties, but jumping a rank for every tie.

> E.g. if there are 3 countries tied for 1st place, they will all receive a rank of 1, and the next country will receive a rank of 4 (not 2). This method is appropriate for our use case as it reflects the presence of ties without artificially influencing the ranks of subsequent countries.

In [33]:
from src.cleaning import add_sample_yearly_rank

df_prepared = add_sample_yearly_rank(df_prepared)

# Now drop happiness_index_rank
df_prepared = df_prepared.drop(columns=["happiness_index_rank"])

for x in df_prepared.columns:
    if df_prepared[x].isna().sum() > 0: 
        print("\nVariable:", x, "\nMissing values:", df_prepared[x].isna().sum())


Variable: undp_developing_regions 
Missing values: 324

Variable: gini_index 
Missing values: 98


In [34]:
df_prepared.info()

<class 'pandas.DataFrame'>
Index: 558 entries, 54 to 1951
Data columns (total 21 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   country                         558 non-null    str    
 1   iso_code                        558 non-null    str    
 2   year                            558 non-null    int64  
 3   co2_per_capita                  558 non-null    float64
 4   consumption_co2_per_capita      558 non-null    float64
 5   temperature_change_from_co2     558 non-null    float64
 6   share_global_co2                558 non-null    float64
 7   land_use_change_co2_per_capita  558 non-null    float64
 8   population                      558 non-null    float64
 9   gdp                             558 non-null    float64
 10  energy_per_capita               558 non-null    float64
 11  renewables_consumption          558 non-null    float64
 12  happiness_index                 558 non-null    fl

In [35]:
display(df_prepared.groupby("year")["happiness_index_rank_62"].count(),
        df_prepared.groupby("year")["happiness_index_rank_62"].agg(["min", "max"]),
        df_prepared.groupby("year")["happiness_index_rank_62"].nunique())

year
2013    62
2014    62
2015    62
2016    62
2017    62
2018    62
2019    62
2020    62
2021    62
Name: happiness_index_rank_62, dtype: int64

,min,max
year,,
2013,1,62
2014,1,62
2015,1,62
2016,1,62
2017,1,62
2018,1,62
2019,1,62
2020,1,62
2021,1,62


year
2013    61
2014    62
2015    62
2016    61
2017    61
2018    61
2019    62
2020    62
2021    61
Name: happiness_index_rank_62, dtype: int64

## 5. Missing values - undp_developing_regions

In [36]:
missing_by_year = (
    df_prepared['undp_developing_regions']
    .isna()
    .groupby(df_raw['year'])
    .sum()
    )

display(missing_by_year)

# Identify the countries with missing values
display(df_prepared[df_prepared['undp_developing_regions'].isna()]['country'].unique())

year
2013    36
2014    36
2015    36
2016    36
2017    36
2018    36
2019    36
2020    36
2021    36
Name: undp_developing_regions, dtype: int64

<StringArray>
[     'Australia',        'Austria',        'Belgium',       'Bulgaria',
         'Canada',    'Switzerland',         'Cyprus',        'Czechia',
        'Germany',        'Denmark',          'Spain',        'Estonia',
        'Finland',         'France', 'United Kingdom',         'Greece',
        'Croatia',        'Hungary',        'Ireland',         'Israel',
          'Italy',          'Japan',    'South Korea',      'Lithuania',
     'Luxembourg',         'Latvia',    'Netherlands',         'Norway',
         'Poland',       'Portugal',        'Romania',         'Russia',
       'Slovakia',       'Slovenia',         'Sweden',  'United States']
Length: 36, dtype: str

The pattern suggests there are 36 countries missing entries for these variables in all years

We import a supplementary json dataset containing the UNDP regions. While it turns out this dataset is missing the desired variable, it does have interesting ways to classify countries whcih were not in the dataset. We add these as follows.

Source: https://github.com/UNDP-Data/Country-Taxonomy

-------
While this dataset has fit nicely into our existing one, it has only added more information, rather than addressing the missing values in undp_developing_regions. We draw from a further dataset to do this:

Source: https://www.kaggle.com/datasets/iamsouravbanerjee/maternal-mortality-dataset

Inspection of the dataset reveals it is missing the same values as we are, so this is just a feature of the classification and data availability as a result. We just fill missing values with 'NOTAPPLICABLE'

In [37]:
# We replace missing values in undp_developing_regions with 'NOTAPPLICABLE'
df_prepared['undp_developing_regions'] = df_prepared['undp_developing_regions'].fillna('NOTAPPLICABLE')

df_prepared['undp_developing_regions'].isna().sum()

np.int64(0)

In [38]:
# We import a supplementary json dataset containing the UNDP regions
from src.io import load_json
from src.config import SUPPLEMENTARY_PATH

un_df = load_json(SUPPLEMENTARY_PATH)
display(un_df.head(100))

,Alpha-3 code-1,Country or Area,Alpha-2 code,Numeric code,Latitude (average),Longitude (average),Group 1,Group 2,Group 3,LDC,LLDC,SIDS,Development classification,Income group
0,GRC,Greece,GR,300,39.0000,22.0000,Europe,Southern Europe,,False,False,False,,High income
1,BRN,Brunei Darussalam,BN,96,4.5000,114.6667,Asia,South-eastern Asia,,False,False,False,,High income
2,BGD,Bangladesh,BD,50,24.0000,90.0000,Asia,Southern Asia,,True,False,False,LDC,Lower middle income
3,CYM,Cayman Islands,KY,136,19.5000,-80.5000,Americas,Latin America and the Caribbean,Caribbean,False,False,False,,High income
4,LBR,Liberia,LR,430,6.5000,-9.5000,Africa,Sub-Saharan Africa,Western Africa,True,False,False,LDC,Low income
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,TUR,Turkey,TR,792,39.0000,35.0000,Asia,Western Asia,,False,False,False,,Upper middle income
96,MAF,Saint Martin (French Part),MF,,18.0708,-63.0501,Americas,Latin America and the Caribbean,Caribbean,False,False,False,,High income
97,SDN,Sudan,SD,729,15.0000,30.0000,Africa,Northern Africa,,True,False,False,LDC,Low income
98,PAN,Panama,PA,591,9.0000,-80.0000,Americas,Latin America and the Caribbean,Central America,False,False,False,,High income


In [39]:
# We apply our previously made function for column name standardization from utils
from src.utils import clean_column_names

un_df = clean_column_names(un_df)

# We rename the Alpha-3 code column to iso_code
un_df = un_df.rename(columns={'alpha_3_code_1': 'iso_code'})
display(un_df)

,iso_code,country_or_area,alpha_2_code,numeric_code,latitude_average,longitude_average,group_1,group_2,group_3,ldc,lldc,sids,development_classification,income_group
0,GRC,Greece,GR,300,39.000000,22.000000,Europe,Southern Europe,,False,False,False,,High income
1,BRN,Brunei Darussalam,BN,96,4.500000,114.666700,Asia,South-eastern Asia,,False,False,False,,High income
2,BGD,Bangladesh,BD,50,24.000000,90.000000,Asia,Southern Asia,,True,False,False,LDC,Lower middle income
3,CYM,Cayman Islands,KY,136,19.500000,-80.500000,Americas,Latin America and the Caribbean,Caribbean,False,False,False,,High income
4,LBR,Liberia,LR,430,6.500000,-9.500000,Africa,Sub-Saharan Africa,Western Africa,True,False,False,LDC,Low income
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234,XKX,Kosovo (as per UNSCR 1244),XK,383,20.872498,42.570787,Europe,Southern Europe,,False,False,False,,
235,PRK,Democratic People's Republic of Korea,KP,408,40.000000,127.000000,Asia,Eastern Asia,,False,False,False,,Low income
236,MAC,"China, Macao Special Administrative Region",MO,446,22.166700,113.550000,Asia,Eastern Asia,,False,False,False,,
237,SHN,Saint Helena,SH,654,-15.933300,-5.700000,Africa,Sub-Saharan Africa,Western Africa,False,False,False,,


In [40]:
print(un_df.columns)

un_df = un_df.rename(columns={
    'group_1': 'continent_un',
    'latitude_average': 'latitude',
    'longitude_average': 'longitude',
    'group_2': 'sub_continent_un',
    'ldc': 'least_devpd_country',
    'lldc': 'landlock_deving_country',
    'sids': 'small_island_deving_country'
    })

un_df = un_df.drop(columns=['development_classification', 'country_or_area', 'alpha_2_code', 'development_classification', 'group_3'])

display(un_df.head())

Index(['iso_code', 'country_or_area', 'alpha_2_code', 'numeric_code',
       'latitude_average', 'longitude_average', 'group_1', 'group_2',
       'group_3', 'ldc', 'lldc', 'sids', 'development_classification',
       'income_group'],
      dtype='str')


,iso_code,numeric_code,latitude,longitude,continent_un,sub_continent_un,least_devpd_country,landlock_deving_country,small_island_deving_country,income_group
0,GRC,300,39.0,22.0000,Europe,Southern Europe,False,False,False,High income
1,BRN,96,4.5,114.6667,Asia,South-eastern Asia,False,False,False,High income
2,BGD,50,24.0,90.0000,Asia,Southern Asia,True,False,False,Lower middle income
3,CYM,136,19.5,-80.5000,Americas,Latin America and the Caribbean,False,False,False,High income
4,LBR,430,6.5,-9.5000,Africa,Sub-Saharan Africa,True,False,False,Low income


In [41]:
# We merge the UNDP regions dataset into our prepared dataset
df_prepared = df_prepared.merge(un_df, on='iso_code', how='left')

display(df_prepared.head())

,country,iso_code,year,co2_per_capita,consumption_co2_per_capita,temperature_change_from_co2,share_global_co2,land_use_change_co2_per_capita,population,gdp,...,happiness_index_rank_62,numeric_code,latitude,longitude,continent_un,sub_continent_un,least_devpd_country,landlock_deving_country,small_island_deving_country,income_group
0,United Arab Emirates,ARE,2013,27.348,33.019,0.002,0.607,-0.014,7831851.0,6.065880e+11,...,11,784,24.0,54.0,Asia,Western Asia,False,False,False,High income
1,United Arab Emirates,ARE,2014,26.045,31.416,0.002,0.605,-0.011,8236880.0,6.318585e+11,...,13,784,24.0,54.0,Asia,Western Asia,False,False,False,High income
2,United Arab Emirates,ARE,2015,25.943,30.139,0.002,0.636,-0.009,8674633.0,6.747427e+11,...,17,784,24.0,54.0,Asia,Western Asia,False,False,False,High income
3,United Arab Emirates,ARE,2016,25.235,28.680,0.002,0.644,-0.007,9030873.0,7.122651e+11,...,22,784,24.0,54.0,Asia,Western Asia,False,False,False,High income
4,United Arab Emirates,ARE,2017,21.279,25.843,0.002,0.546,-0.007,9234330.0,7.175002e+11,...,18,784,24.0,54.0,Asia,Western Asia,False,False,False,High income


In [42]:
df_prepared.info()

<class 'pandas.DataFrame'>
RangeIndex: 558 entries, 0 to 557
Data columns (total 30 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   country                         558 non-null    str    
 1   iso_code                        558 non-null    str    
 2   year                            558 non-null    int64  
 3   co2_per_capita                  558 non-null    float64
 4   consumption_co2_per_capita      558 non-null    float64
 5   temperature_change_from_co2     558 non-null    float64
 6   share_global_co2                558 non-null    float64
 7   land_use_change_co2_per_capita  558 non-null    float64
 8   population                      558 non-null    float64
 9   gdp                             558 non-null    float64
 10  energy_per_capita               558 non-null    float64
 11  renewables_consumption          558 non-null    float64
 12  happiness_index                 558 non-null   

### 5.1 Competing columns - continent vs continent_un

In [43]:
# Note, this resulted in two continent columns
# Compare value counts
print("\ncontinent value counts:")
print(df_prepared['continent'].value_counts(dropna=False))
print("\ncontinent_un value counts:")
print(df_prepared['continent_un'].value_counts(dropna=False))


continent value counts:
continent
Europe     279
Asia       162
America     81
Africa      27
Oceania      9
Name: count, dtype: int64

continent_un value counts:
continent_un
Europe      279
Asia        162
Americas     81
Africa       27
Oceania       9
Name: count, dtype: int64


In [44]:
# So they are the same except continent_un has Americas instead of America - this is just convention, here we will keep continent, which has it without the s:
df_prepared = df_prepared.drop(columns=['continent_un'])
df_prepared.columns

Index(['country', 'iso_code', 'year', 'co2_per_capita',
       'consumption_co2_per_capita', 'temperature_change_from_co2',
       'share_global_co2', 'land_use_change_co2_per_capita', 'population',
       'gdp', 'energy_per_capita', 'renewables_consumption', 'happiness_index',
       'continent', 'hemisphere', 'human_development_groups', 'hdi_rank_2021',
       'undp_developing_regions', 'material_footprint_per_capita',
       'gini_index', 'happiness_index_rank_62', 'numeric_code', 'latitude',
       'longitude', 'sub_continent_un', 'least_devpd_country',
       'landlock_deving_country', 'small_island_deving_country',
       'income_group'],
      dtype='str')

## 6. Missing values - gini_index

This is mostly a control variable for income inequality.

In [45]:
# Check for remaining missing values

for x in df_prepared.columns:
    if (df_prepared[x].isna().sum() > 0) & (x != 'undp_developing_regions'): 
        print("\nVariable:", x, "\nMissing values:", df_prepared[x].isna().sum())


Variable: gini_index 
Missing values: 98


In [46]:
missing_by_year = (
    df_prepared['gini_index']
    .isna()
    .groupby(df_raw['year'])
    .sum()
    )

display(missing_by_year)

# Identify the countries with missing values
imputed_gini = df_prepared[df_prepared['gini_index'].isna()]['country'].unique()
display(imputed_gini)



year
2013.0    6
2014.0    4
2015.0    2
2016.0    3
2017.0    1
2018.0    4
2019.0    1
2020.0    3
2021.0    3
Name: gini_index, dtype: int64

<StringArray>
['United Arab Emirates',            'Argentina',            'Australia',
           'Bangladesh',              'Belarus',                'Chile',
                'Egypt',              'Hungary',                'Japan',
          'South Korea',            'Sri Lanka',              'Morocco',
               'Mexico',             'Malaysia',             'Pakistan',
          'Philippines',                'Qatar',              'Ukraine',
              'Vietnam',         'South Africa']
Length: 20, dtype: str

In [47]:
df_prepared['year'].dtype

dtype('int64')

While there are 20 countries missing some gini_index values, all countries remaining in the dataset have at least 3 gini_index entries.

This is also different to the happiness_index in QAT case which was specifically missing values in 2020 and 2021. Here, missing values are more prevalent and distributed accross 20 different countries.

Linear interpolation is used to fill years with available values around them and endpoints are filled with the same value as their neighbouring entry - this is just the limit_direction=`both' setting in the interpolate function.

We note, that interpretations of changes in gini_index nearing more recent years should be done with care, and paying particular attention to the following countries. Other types of analyses and controlling for income inequality are still possible.

We store the list of affected countries in `imputed_gini` to allow for analyses excluding these for reliability checkas and more in-depth-analyses on the effects of income inequality.

In [48]:
# While there are 20 countries missing some gini_index values, all countries remaining in the dataset have at least 3 gini_index entries.

# Linear interpolation is used to fill years with available values around them, endpoints are then filled with the same value as their neighbouring entry - this is just the limit_direction=`both' setting.
df_prepared['gini_index'] = df_prepared.groupby('country')['gini_index'].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
print(f"There are {df_prepared['gini_index'].isna().sum()} missing values remaining in gini_index column after linear interpolation")


There are 0 missing values remaining in gini_index column after linear interpolation


In [49]:
missing_by_year = (
    df_prepared['gini_index']
    .isna()
    .groupby(df_prepared['year'])
    .sum()
    )

display(missing_by_year)

year
2013    0
2014    0
2015    0
2016    0
2017    0
2018    0
2019    0
2020    0
2021    0
Name: gini_index, dtype: int64

In [50]:
# Check for remaining missing values

for x in df_prepared.columns:
    if (df_prepared[x].isna().sum() > 0) & (x != 'undp_developing_regions'): 
        print("\nVariable:", x, "\nMissing values:", df_prepared[x].isna().sum())
    elif (df_prepared[x].isna().sum() > 0) | (x == 'undp_developing_regions'):
        print("There are no missing values remaining other than the undp_developing_regions variable")

There are no missing values remaining other than the undp_developing_regions variable


In [51]:
index_vars = ['numeric_code', 'iso_code', 'country', 'year']
country_group_vars = ['human_development_groups', 'hdi_rank_2021', 'undp_developing_regions', 'least_devpd_country', 'landlock_deving_country', 'small_island_deving_country']
impact_vars = ['co2_per_capita', 'consumption_co2_per_capita', 'material_footprint_per_capita', 'energy_per_capita', 'renewables_consumption', 'temperature_change_from_co2', 'share_global_co2', 'land_use_change_co2_per_capita']
socioeconomic_vars = ['population', 'gdp', 'happiness_index', 'gini_index', 'income_group', 'happiness_index_rank_62']
location_vars = ['continent', 'sub_continent_un', 'hemisphere', 'latitude', 'longitude']

# We use the above groupings to reorder the columns in the dataset, and we order by iso_code and year
df_clean = df_prepared[index_vars + country_group_vars + socioeconomic_vars + impact_vars + location_vars]
df_clean = df_clean.sort_values(by=['iso_code', 'year'])
df_clean = df_clean.reset_index(drop=True)
df_clean.head()



,numeric_code,iso_code,country,year,human_development_groups,hdi_rank_2021,undp_developing_regions,least_devpd_country,landlock_deving_country,small_island_deving_country,...,energy_per_capita,renewables_consumption,temperature_change_from_co2,share_global_co2,land_use_change_co2_per_capita,continent,sub_continent_un,hemisphere,latitude,longitude
0,784,ARE,United Arab Emirates,2013,Very High,26.0,AS,False,False,False,...,144520.031,0.222,0.002,0.607,-0.014,Asia,Western Asia,Northern Hemisphere,24.0,54.0
1,784,ARE,United Arab Emirates,2014,Very High,26.0,AS,False,False,False,...,137818.312,0.792,0.002,0.605,-0.011,Asia,Western Asia,Northern Hemisphere,24.0,54.0
2,784,ARE,United Arab Emirates,2015,Very High,26.0,AS,False,False,False,...,140994.875,0.761,0.002,0.636,-0.009,Asia,Western Asia,Northern Hemisphere,24.0,54.0
3,784,ARE,United Arab Emirates,2016,Very High,26.0,AS,False,False,False,...,140575.797,0.794,0.002,0.644,-0.007,Asia,Western Asia,Northern Hemisphere,24.0,54.0
4,784,ARE,United Arab Emirates,2017,Very High,26.0,AS,False,False,False,...,131237.281,1.863,0.002,0.546,-0.007,Asia,Western Asia,Northern Hemisphere,24.0,54.0


In [52]:
# Confirm no columns were dropped in the reordering process

print(df_prepared.shape, df_clean.shape)
print(set(df_prepared.columns) - set(df_clean.columns))

(558, 29) (558, 29)
set()


## 7. Potentially wrong dtypes - hdi_rank_2021, population

It makes sense to expect both these variables to be integers, since neither ranks nor people can in principle contain decimals.

In [53]:
df_clean[['hdi_rank_2021', 'population']].dtypes

hdi_rank_2021    float64
population       float64
dtype: object

We check whether there are any non-zero decimals in either variables before transforming them to integers.

In [54]:
for col in ['hdi_rank_2021', 'population']:
    # Find rows where value is float AND not equal to its integer conversion
    mask = df_clean[col].apply(lambda x: isinstance(x, float) and x != int(x))
    problematic = df_clean[mask][col].unique()
    
    if len(problematic) > 0:
        print(f"\n{col}: {problematic}")
    elif len(problematic) == 0:
        print(f"\nNo problematic values found in {col} column")


No problematic values found in hdi_rank_2021 column

No problematic values found in population column


In [55]:
df_clean['hdi_rank_2021'] = df_clean['hdi_rank_2021'].astype(int)
df_clean['population'] = df_clean['population'].astype(int)
df_clean[['hdi_rank_2021', 'population']].dtypes



hdi_rank_2021    int64
population       int64
dtype: object

In [56]:
# Quick summary
print(f"The resulting data has {df_clean.shape[0]} rows and {df_clean.shape[1]} columns")
print(f"\nMissing values total: {df_clean.isna().sum().sum()}")
print(f"\nColumn data types:\n{df_clean.info()}")
# display(df_clean)

# Only show describe for numeric columns
numeric_cols = df_clean.select_dtypes(include=['number']).columns
if len(numeric_cols) > 0:
    print("\nDescriptive statistics for numeric columns:\n")
    display(df_clean[numeric_cols].describe().round(2))

The resulting data has 558 rows and 29 columns

Missing values total: 0
<class 'pandas.DataFrame'>
RangeIndex: 558 entries, 0 to 557
Data columns (total 29 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   numeric_code                    558 non-null    str    
 1   iso_code                        558 non-null    str    
 2   country                         558 non-null    str    
 3   year                            558 non-null    int64  
 4   human_development_groups        558 non-null    str    
 5   hdi_rank_2021                   558 non-null    int64  
 6   undp_developing_regions         558 non-null    str    
 7   least_devpd_country             558 non-null    bool   
 8   landlock_deving_country         558 non-null    bool   
 9   small_island_deving_country     558 non-null    bool   
 10  population                      558 non-null    int64  
 11  gdp                             558 

,year,hdi_rank_2021,population,gdp,happiness_index,gini_index,happiness_index_rank_62,co2_per_capita,consumption_co2_per_capita,material_footprint_per_capita,energy_per_capita,renewables_consumption,temperature_change_from_co2,share_global_co2,land_use_change_co2_per_capita,latitude,longitude
count,558.00,558.00,5.580000e+02,5.580000e+02,558.00,558.00,558.00,558.00,558.00,558.00,558.00,558.00,558.00,558.00,558.00,558.00,558.00
mean,2017.00,49.50,7.381144e+07,1.515797e+12,6.09,34.99,31.49,7.24,8.07,21.28,38867.61,245.35,0.01,1.32,0.66,33.16,23.97
std,2.58,36.91,1.838700e+08,3.433515e+12,0.91,7.12,17.91,6.02,5.57,13.03,34234.54,649.87,0.03,4.01,1.59,24.81,57.73
min,2013.00,1.00,5.433580e+05,2.447558e+10,3.98,23.20,1.00,0.41,0.41,2.15,1958.55,0.22,0.00,0.02,-2.33,-34.00,-102.00
25%,2015.00,21.00,8.511636e+06,2.565229e+11,5.39,30.30,16.00,3.98,4.47,10.82,20571.47,19.38,0.00,0.12,-0.11,24.00,4.00
50%,2017.00,39.50,2.296832e+07,4.870433e+11,6.01,33.85,31.50,5.95,7.07,18.28,30956.46,66.34,0.00,0.28,0.16,39.75,21.00
75%,2019.00,76.00,7.108510e+07,1.413633e+12,6.89,38.60,47.00,8.44,10.08,29.81,45800.55,158.47,0.01,0.97,0.99,50.83,53.00
max,2021.00,161.00,1.426437e+09,2.618060e+13,7.84,59.60,62.00,44.98,33.02,75.61,245295.05,6170.01,0.24,30.99,8.57,64.00,138.00


## 8. Store the final dataset

In [57]:
from src.config import CLEAN_PATH
from src.io import save_csv

save_csv(df_clean, CLEAN_PATH)

# Dataset Overview
## *Sustainability, Wellbeing and Resource Use Dataset*

- **Rows:** 558  
- **Columns:** 29  
- **Total Missing Values:** 0

| Data Type | Count |
|---|---:|
| `str` | 9 |
| `int64` | 4 |
| `float64` | 13 |
| `bool` | 3 |

---

## Variable Dictionary by type

### 1. Index Variables

| Variable | Data Type | Description |
|---|---|---|
| `numeric_code` | `str` | Numeric country identifier code, typically aligned with ISO numeric standards. |
| `iso_code` | `str` | ISO alpha country code (e.g., ESP, USA, FRA). |
| `country` | `str` | Country name. |
| `year` | `int64` | Observation year associated with the record. |

---

### 2. Country Group Variables

| Variable | Data Type | Description |
|---|---|---|
| `human_development_groups` | `str` | Human Development Index (HDI) classification group (e.g., Low, Medium, High, Very High). |
| `hdi_rank_2021` | `int64` | HDI rank based on the 2021 Human Development Report. Lower values indicate stronger development performance. |
| `undp_developing_regions` | `str` | UNDP regional classification for developing economies. Contains missing values. |
| `least_devpd_country` | `bool` | Indicates whether the country is classified as a Least Developed Country (LDC). |
| `landlock_deving_country` | `bool` | Indicates whether the country is classified as a Landlocked Developing Country (LLDC). |
| `small_island_deving_country` | `bool` | Indicates whether the country is classified as a Small Island Developing State (SIDS). |

---

### 3. Impact Variables

| Variable | Data Type | Description |
|---|---|---|
| `co2_per_capita` | `float64` | Carbon dioxide emissions per capita. |
| `consumption_co2_per_capita` | `float64` | Consumption-based CO₂ emissions per capita, accounting for trade-adjusted emissions. |
| `material_footprint_per_capita` | `float64` | Per-capita material footprint associated with domestic consumption. |
| `energy_per_capita` | `float64` | Energy consumption per capita. |
| `renewables_consumption` | `float64` | Renewable energy consumption share or level. |
| `temperature_change_from_co2` | `float64` | Estimated temperature contribution associated with CO₂ emissions. |
| `share_global_co2` | `float64` | Country share of global CO₂ emissions. |
| `land_use_change_co2_per_capita` | `float64` | Per-capita CO₂ emissions resulting from land-use changes. |

---

### 4. Socioeconomic Variables

| Variable | Data Type | Description |
|---|---|---|
| `population` | `int64` | Total population of the country in the given year. |
| `gdp` | `float64` | Gross Domestic Product (GDP), likely measured in USD. |
| `happiness_index` | `float64` | National happiness or subjective wellbeing score. |
| `gini_index` | `float64` | Income inequality measure; higher values indicate greater inequality. |
| `income_group` | `str` | Income classification category (e.g., low income, upper middle income). |
| `happiness_index_rank_62` | `int64` | Country ranking based on happiness index among sampled countries. |

---

### 5. Location Variables

| Variable | Data Type | Description |
|---|---|---|
| `continent` | `str` | Continent classification of the country. |
| `sub_continent_un` | `str` | United Nations sub-region classification. |
| `hemisphere` | `str` | Geographic hemisphere classification. |
| `latitude` | `float64` | Latitude coordinate of the country reference point or centroid. |
| `longitude` | `float64` | Longitude coordinate of the country reference point or centroid. |

---

## Missing Values Summary

| Variable | Missing Values | Notes |
|---|---:|---|
| `undp_developing_regions` | 324 | Missing for countries outside UNDP developing-region classifications. |